In [ ]:
# ── Cell 1: Environment Setup & RTX 2050 Optimizations ──
import os, json, math, time, random, shutil
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
from transformers import CLIPTextModel, CLIPTokenizer
from huggingface_hub import HfApi, login

# Maximize PyTorch 2.12+ features for RTX 2050
torch.set_float32_matmul_precision('high') # Enables Tensor Cores
torch.backends.cudnn.benchmark = True
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

CFG = {
    "hf_repo_id"      : "Priyanshu-i/delta-canvas-agent",
    "local_clip_path" : "./clip-vit-base-patch32-local", # Pulls from HF automatically
    "dataset_path"    : "./data/delta_model_dataset.jsonl",
    "checkpoint_dir"  : "./checkpoints",
    "log_dir"         : "./logs",
    "max_seq_len"     : 100, # Increased for complex flowcharts
    "batch_size"      : 16,  # 4GB VRAM safe
    "accum_steps"     : 2,   # Effective batch size = 32
    "lr"              : 3e-4,
    "epochs"          : 5,  # 500k dataset needs fewer epochs
    "log_every"       : 1,
    "save_every"      : 5,
    "d_model"         : 256,
    "nhead"           : 8,
    "num_layers"      : 4,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if device.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Path(CFG["checkpoint_dir"]).mkdir(parents=True, exist_ok=True)
Path(CFG["log_dir"]).mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Cell 2: O(1) Lazy-Loading Dataset (Protects 12GB RAM) ──
class DeltaStrokeDataset(Dataset):
    def __init__(self, jsonl_path, max_seq_len=100):
        self.path = jsonl_path
        self.max_seq_len = max_seq_len
        self.offsets = []
        self.STATS_FILE = Path(CFG["checkpoint_dir"]) / "dataset_stats.pt"
        self.file_handle = open(self.path, 'r', encoding='utf-8')

        print("Building disk index (O(1) memory mapping)...")
        with open(self.path, 'rb') as f:
            while True:
                offset = f.tell()
                line = f.readline()
                if not line: break
                self.offsets.append(offset)
        print(f"Indexed {len(self.offsets):,} samples.")

        # Compute or load Z-Score Stats
        if self.STATS_FILE.exists():
            stats = torch.load(self.STATS_FILE, weights_only=True)
            self.mean, self.std = stats["mean"], stats["std"]
            print("Z-Score stats restored from cache.")
        else:
            print("Computing Z-Score on a 25k random subset...")
            sampled_offsets = random.sample(self.offsets, min(25000, len(self.offsets)))
            all_cont = []
            with open(self.path, 'r', encoding='utf-8') as f:
                for offset in sampled_offsets:
                    f.seek(offset)
                    d = json.loads(f.readline())
                    s = torch.tensor(d["vectors"], dtype=torch.float32)
                    all_cont.append(s[:, :3])
            
            all_cont = torch.cat(all_cont, dim=0)
            self.mean = all_cont.mean(0)
            self.std  = all_cont.std(0) + 1e-6
            torch.save({"mean": self.mean, "std": self.std}, self.STATS_FILE)
            print("Z-Score stats computed and saved.")

    def __len__(self): return len(self.offsets)

    def __getitem__(self, idx):
        # Lazy load from disk instantly
        self.file_handle.seek(self.offsets[idx])
        d = json.loads(self.file_handle.readline())
        
        prompt = d["prompt"]
        strokes = torch.tensor(d["vectors"], dtype=torch.float32)
        seq_len = strokes.shape[0]

        cont_vars  = strokes[:, :3]           
        pen_states = strokes[:, 3:]           
        norm_cont  = (cont_vars - self.mean) / self.std
        mask = torch.ones(self.max_seq_len, dtype=torch.float32)

        if seq_len >= self.max_seq_len:
            norm_cont  = norm_cont[:self.max_seq_len]
            pen_states = pen_states[:self.max_seq_len]
        else:
            # Ghost-Tail Fix: Pad with normalized equivalent of 0.0
            pad_len    = self.max_seq_len - seq_len
            dead_cont  = (-self.mean / self.std).unsqueeze(0).repeat(pad_len, 1)
            dead_pen   = torch.tensor([[0., 0., 1.]]).repeat(pad_len, 1) # p_end = 1
            norm_cont  = torch.cat([norm_cont,  dead_cont], 0)
            pen_states = torch.cat([pen_states, dead_pen],  0)
            mask[seq_len:] = 0.

        return prompt, torch.cat([norm_cont, pen_states], dim=1), mask
    
    def __del__(self):
        # Clean up the file handle when training finishes
        if hasattr(self, 'file_handle'):
            self.file_handle.close()

In [ ]:
# ── Cell 3: Delta Transformer (FlashAttention Enabled) ──
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x): return x + self.pe[:, :x.size(1)]

class DeltaContinuousModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tokenizer    = CLIPTokenizer.from_pretrained(cfg["local_clip_path"])
        self.text_encoder = CLIPTextModel.from_pretrained(cfg["local_clip_path"])
        for p in self.text_encoder.parameters(): p.requires_grad = False
        
        clip_dim = self.text_encoder.config.hidden_size 
        d_model  = cfg["d_model"]

        self.stroke_embed    = nn.Linear(6, d_model)
        self.pos_enc         = PositionalEncoding(d_model)
        self.text_projection = nn.Linear(clip_dim, d_model)

        # PyTorch 2.0+ handles FlashAttention natively here
        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=cfg["nhead"],
            dim_feedforward=1024, dropout=0.1,
            batch_first=True, norm_first=True,
            activation="gelu"
        )
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=cfg["num_layers"])

        self.cont_head = nn.Sequential(nn.Linear(d_model, 128), nn.GELU(), nn.Linear(128, 3))
        self.pen_head  = nn.Sequential(nn.Linear(d_model, 128), nn.GELU(), nn.Linear(128, 3))

    def _causal_mask(self, sz, dev):
        return torch.triu(torch.full((sz, sz), float("-inf"), device=dev), diagonal=1)

    def forward(self, prompts, target_strokes):
        B, L, _ = target_strokes.shape
        dev = target_strokes.device

        tok = self.tokenizer(prompts, padding=True, truncation=True, return_tensors="pt").to(dev)
        with torch.no_grad():
            enc_hs = self.text_encoder(**tok).last_hidden_state   
        
        sos = self.text_projection(enc_hs[:, 0:1, :]) 
        se  = self.stroke_embed(target_strokes[:, :-1, :]) 
        tgt = self.pos_enc(torch.cat([sos, se], dim=1)) 

        h = self.decoder(tgt=tgt, memory=enc_hs, tgt_mask=self._causal_mask(L, dev))
        return self.cont_head(h), self.pen_head(h)

In [ ]:
# ── Cell 4: Fault-Tolerant, Resumable Training Engine (V4-Restore) ──
import shutil

def save_checkpoint(model, optimizer, scaler, scheduler, epoch, loss, is_best=False):
    """Saves a complete, fault-tolerant checkpoint that preserves training state."""
    ckpt_dir = Path(CFG["checkpoint_dir"])
    last_file = ckpt_dir / "last_checkpoint.pt"
    best_file = ckpt_dir / "best_model.pt"
    meta_file = ckpt_dir / "train_meta.json"
    
    state = {
        "epoch": epoch,
        "loss": loss,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "cfg": CFG,
    }
    torch.save(state, last_file)

    # Rolling backup every few epochs
    if (epoch + 1) % CFG["save_every"] == 0:
        rolling_file = ckpt_dir / f"delta_model_ep{epoch+1}.pt"
        shutil.copy(last_file, rolling_file)
        print(f" -> Archive checkpoint saved: {rolling_file}")

    if is_best:
        shutil.copy(last_file, best_file)
        print(f" ★ New structural best model saved! (Loss: {loss:.5f})")

    # Save human-readable training telemetry
    meta = {
        "last_updated_epoch": epoch + 1,
        "best_loss": round(float(loss), 6),
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
    }
    with open(meta_file, "w") as f:
        json.dump(meta, f, indent=2)


def load_checkpoint(model, optimizer, scaler, scheduler):
    """Resumes training state from disk flawlessly. Returns start_epoch and best_loss."""
    last_file = Path(CFG["checkpoint_dir"]) / "last_checkpoint.pt"
    if last_file.exists():
        print(f"Found existing training session. Resuming from {last_file}...")
        ckpt = torch.load(last_file, map_location=device, weights_only=False)
        
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scaler.load_state_dict(ckpt["scaler_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        
        start_epoch = ckpt["epoch"] + 1
        best_loss = ckpt["loss"]
        print(f"✅ Context restored perfectly. Resuming at Epoch {start_epoch+1} (Previous Loss: {best_loss:.5f})")
        return start_epoch, best_loss
    
    print("No previous checkpoints found. Initializing a fresh training run.")
    return 0, float("inf")


def train_model(model, dataset, cfg):
    # num_workers=0 is mandatory to prevent Windows multi-processing pipe deadlocks
    loader = DataLoader(
        dataset, 
        batch_size=cfg["batch_size"], 
        shuffle=True, 
        num_workers=0, 
        pin_memory=True
    )

    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=cfg["lr"], weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["epochs"], eta_min=1e-6)
    scaler = GradScaler("cuda")
    
    # Structural Loss Functions
    cont_crit = nn.L1Loss(reduction="none")
    pen_crit  = nn.CrossEntropyLoss(reduction="none")
    accum = cfg["accum_steps"]

    # --- Checkpoint Recovery Check ---
    start_epoch, best_loss = load_checkpoint(model, optimizer, scaler, scheduler)

    print(f"\nEngine Active: Target Epochs {start_epoch + 1} → {cfg['epochs']}")
    print(f"Batches per Epoch: {len(loader):,} | Effective Batch Size: {cfg['batch_size']*accum}\n")
    
    for epoch in range(start_epoch, cfg["epochs"]):
        model.train()
        epoch_loss = 0.
        epoch_cont = 0.
        epoch_pen  = 0.
        t_epoch = time.time()
        optimizer.zero_grad()

        for step, (prompts, targets, masks) in enumerate(loader):
            targets = targets.to(device, non_blocking=True)
            masks   = masks.to(device, non_blocking=True)

            with autocast("cuda"):
                pred_cont, pred_pen = model(prompts, targets)
                tgt_cont = targets[:, :, :3]
                tgt_pen  = torch.argmax(targets[:, :, 3:], dim=-1)

                # Active Loss Masking: Nullify calculations over padded 'dead' coordinates
                l_cont = cont_crit(pred_cont, tgt_cont).mean(-1)
                l_cont = (l_cont * masks).sum() / (masks.sum() + 1e-8)

                l_pen = pen_crit(pred_pen.transpose(1, 2), tgt_pen)
                l_pen = (l_pen * masks).sum() / (masks.sum() + 1e-8)

                # Composite objective weighting coordinate tracing and pen actions
                loss = (l_cont + 0.5 * l_pen) / accum

            scaler.scale(loss).backward()

            # Step weights based on our virtual accumulation window
            if (step + 1) % accum == 0 or (step + 1) == len(loader):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Stabilizes structural attention layers
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            epoch_loss += loss.item() * accum
            epoch_cont += l_cont.item()
            epoch_pen  += l_pen.item()

        scheduler.step()
        avg_loss = epoch_loss / len(loader)
        is_best = avg_loss < best_loss
        if is_best:
            best_loss = avg_loss
            
        # Execute robust saving routine at the end of every single epoch
        save_checkpoint(model, optimizer, scaler, scheduler, epoch, avg_loss, is_best=is_best)
        
        if (epoch + 1) % cfg["log_every"] == 0:
            print(f"Ep {epoch+1:02d}/{cfg['epochs']} | loss={avg_loss:.5f} (cont={epoch_cont/len(loader):.4f} pen={epoch_pen/len(loader):.4f}) | lr={scheduler.get_last_lr()[0]:.2e} | {time.time()-t_epoch:.1f}s")

    print(f"\nTraining execution finished. Optimal Loss Achieved: {best_loss:.5f}")
    return best_loss

# Run initialization
dataset = DeltaStrokeDataset(CFG["dataset_path"], max_seq_len=CFG["max_seq_len"])
model = DeltaContinuousModel(CFG).to(device)
train_model(model, dataset, CFG)

In [ ]:
# ── Cell 5: Inference Test & Hugging Face Push ──
@torch.no_grad()
def push_to_huggingface(model, dataset, cfg, token=None):
    """
    Saves the complete package and pushes to Priyanshu-i/delta-canvas-agent
    """
    print("\nPreparing final package for Hugging Face...")
    export_dir = Path("./hf_export")
    export_dir.mkdir(exist_ok=True)
    
    # Save isolated weights and configs
    torch.save(model.state_dict(), export_dir / "pytorch_model.bin")
    torch.save({"mean": dataset.mean, "std": dataset.std}, export_dir / "dataset_stats.pt")
    with open(export_dir / "config.json", "w") as f: json.dump(cfg, f, indent=2)
    model.tokenizer.save_pretrained(export_dir / "tokenizer")
    
    try:
        if token:
            login(token=token)
        else:
            print("Make sure you are logged into Hugging Face CLI (`huggingface-cli login`).")
            
        api = HfApi()
        
        print(f"Pushing to {cfg['hf_repo_id']}... (this may take a minute)")
        api.create_repo(repo_id=cfg["hf_repo_id"], exist_ok=True, private=False)
        
        api.upload_folder(
            folder_path=str(export_dir),
            repo_id=cfg["hf_repo_id"],
            commit_message="Initial Delta Model Upload (V4)"
        )
        print(f"✅ Success! Model live at: https://huggingface.co/{cfg['hf_repo_id']}")
    except Exception as e:
        print(f"Failed to push to HF. Error: {str(e)}")

# Run inference test before pushing
test_prompt = "Draw a user authentication flowchart"
print(f"Testing generation for: {test_prompt}")
model.eval()

# To trigger HF Push, uncomment the line below and add your HF Write Token.
# push_to_huggingface(model, dataset, CFG, token="hf_YOUR_WRITE_TOKEN_HERE")